# What a CRISM band is

A CRISM observation arrives as a cube of numbers with no wavelengths anywhere in it. The
label says the file holds 55 bands and says nothing about what any of those 55 bands
looked at. Band 0 of one observation and band 0 of the next need not be the same colour
of light.

## How CRISM creates data

CRISM scans Mars one thin line at a time, just like a document scanner. As the spacecraft flies forward, light from a single strip of ground enters the camera. Inside, a glass grating splits that light into a rainbow across a rectangular sensor:

* **Columns (left to right):** Each column is a specific spot on the ground.
* **Rows (top to bottom):** Each row is a specific color of light.

Stacking these snapshots over time creates a 3D data file made of three parts:

* **Lines (Time & Distance):** The forward movement of the spacecraft along its flight path over time.
* **Samples (Position):** The location of spots going across the scanned line.
* **Bands (Sensor Rows):** The individual rows on the sensor. A row's assigned color depends on how the internal optics split the light.

CRISM uses two separate sensors that save into separate files:

* **`S` sensor:** Captures visible light.
* **`L` sensor:** Captures infrared light.

Because each sensor records a different set and number of color rows, every single scan generates two 3D data files of different sizes.

## Setup

In [1]:
"""Bring one observation down and name what to look at inside it."""

from pathlib import Path

import numpy as np

from preprocessing.crism.calibration import bands_calibration, wavelengths
from preprocessing.crism.fetching import download
from preprocessing.crism.storage import locations, naming, reading

OBSERVATION = "msp00006994_05_if214_trr3"

LINE, SAMPLE = 1000, 32

CDR_ROOT = Path(wavelengths.__file__).parent / "cdr"

for detector, label in download.fetch(OBSERVATION).items():
    print(detector, label.name)

l msp00006994_05_if214l_trr3.lbl
s msp00006994_05_if214s_trr3.lbl


In [2]:
"""Two printers, so every table below is laid out the same way."""


def edges(size, keep=3):
    """Return the first and last few indices of an axis.

    Args:
        size: How long the axis is.
        keep: How many indices to take from each end.

    Returns:
        The indices, first end then last.
    """
    return list(range(keep)) + list(range(size - keep, size))


def block(grid, rows, cols, digits=2):
    """Print part of a two dimensional grid as a plain table.

    Args:
        grid: The grid to read from.
        rows: Which row indices to print, in the order to print them.
        cols: Which column indices to print, in the order to print them.
        digits: How many decimals to show each value with.

    Returns:
        None.
    """
    print("      " + "".join(f"{col:>10}" for col in cols))
    for row in rows:
        cells = "".join(f"{grid[row, col]:>10.{digits}f}" for col in cols)
        print(f"{row:>6}" + cells)

## The TRDR

The label is the only description the file carries of its own shape. It gives the three
axis lengths, the order the values were written in, and the width of one number. It does
not give a single wavelength.

`PIXEL_AVERAGING_WIDTH` is the binning: 10 detector columns per sample, which is where 64
samples comes from.

In [3]:
"""What each detector's label says about the shape of its cube."""

head = f"{'det':>4}{'lines':>8}{'samples':>9}{'bands':>7}{'bin':>5}"
print(f"{head}  {'order':<19}unit")
for name in naming.DETECTORS:
    image = locations.files(OBSERVATION, name)[".img"]
    label = reading.load_label(image.with_suffix(".lbl"))
    lines, samples, bands, stored, dtype = reading.load_layout(label)
    row = f"{name:>4}{lines:>8}{samples:>9}{bands:>7}"
    print(f"{row}{label['PIXEL_AVERAGING_WIDTH']:>5}  {stored:<19}{label['UNIT']}")

 det   lines  samples  bands  bin  order              unit
   l    2700       64     55   10  LINE_INTERLEAVED   I_OVER_F
   s    2700       64     19   10  LINE_INTERLEAVED   I_OVER_F


### One band of the raw cube

The file writes `65535.0` there, which is a flag meaning the detector was never calibrated at that column.
Left as a number it is roughly two hundred thousand times a real value, so any average that includes it is ruined.

In [16]:
"""The infrared cube exactly as the file writes it, at one band."""

image = locations.files(OBSERVATION, "l")[".img"]
label = reading.load_label(image.with_suffix(".lbl"))
raw = reading.build_cube(image, label)

print(f"cube {raw.shape}, band fixed 10, lines rows, samples columns\n")
block(raw[:, :, 10], edges(raw.shape[0]), edges(raw.shape[1]), digits=3)

cube (2700, 64, 55), band fixed 10, lines rows, samples columns

               0         1         2        61        62        63
     0 65535.000 65535.000 65535.000     0.173     0.145 65535.000
     1 65535.000 65535.000 65535.000     0.169     0.142 65535.000
     2 65535.000 65535.000 65535.000     0.167     0.143 65535.000
  2697 65535.000 65535.000 65535.000     0.183     0.154 65535.000
  2698 65535.000 65535.000 65535.000     0.186     0.157 65535.000
  2699 65535.000 65535.000 65535.000     0.185     0.156 65535.000


### One pixel of the raw cube

Fixing a line and a sample and reading across the bands gives a spectrum. This is what the
file offers: a value per band index, and no way to know what band index means.

Band 0 is `65535.0` down the whole cube. That row of the detector was downlinked but never
calibrated, so the entire band is a flag.

In [5]:
"""The spectrum the file gives at one pixel, indexed only by band number."""

print(f"line {LINE}, sample {SAMPLE}, {raw.shape[2]} bands")
print("".join(f"{band:>10}" for band in edges(raw.shape[2], 5)))
print("".join(f"{raw[LINE, SAMPLE, band]:>10.3f}" for band in edges(raw.shape[2], 5)))

line 1000, sample 32, 55 bands
         0         1         2         3         4        50        51        52        53        54
 65535.000     0.043     0.130     0.158     0.147     0.230     0.243     0.245     0.236     0.250


## The CDR
A Calibration Data Record (CDR) is a pre-launch reference file that describes the instrument rather than a specific observation.

The WA record is the wavelength CDR. It maps every detector column and row to its center wavelength in nanometers. It is a 2D table because the instrument projects light in a slight curve across the sensor. This optical effect, called spectral smile, causes the exact wavelength of a single row to shift from left to right.

Observation labels state which WA record to use to look up these wavelengths.

In [7]:
"""Which wavelength record each detector of this observation names."""

for name in naming.DETECTORS:
    label = reading.load_label(locations.files(OBSERVATION, name)[".lbl"])
    print(f"{name}  bands {label['BANDS']:>3}  {label['MRO:WAVELENGTH_FILE_NAME']}")

l  bands  55  CDR410803692813_WA0300010L_3.IMG
s  bands  19  CDR420850327000_WA0300010S_2.IMG


### The record as bytes, then as the loader reads it

The `.img` of a CDR is built the same way an observation is: a plain grid of 32 bit floats
with a label beside it. It holds one line, 64 columns and one value per band, and it writes
the same `65535` flag where nothing was ever calibrated.

The first table is the file untouched. The second is what `wavelengths.load` returns, which
is the same grid with the flag turned into `NaN` so that averaging it is impossible rather
than merely wrong.

In [9]:
"""The infrared record at 55 bands, as written and as loaded."""

stored = np.fromfile(CDR_ROOT / "infrared_55.img", dtype="<f4", count=64 * 55)
stored = stored.reshape(55, 64).T
loaded = wavelengths.load("l", 55)

print("as written, columns down, bands across")
block(stored, edges(64), edges(55), digits=1)
print()
print("as loaded")
block(loaded, edges(64), edges(55), digits=1)

as written, columns down, bands across
               0         1         2        52        53        54
     0   65535.0   65535.0   65535.0   65535.0   65535.0   65535.0
     1   65535.0   65535.0   65535.0   65535.0   65535.0   65535.0
     2   65535.0   65535.0   65535.0   65535.0   65535.0   65535.0
    61   65535.0    3929.0    3761.0    1085.2    1052.4    1026.1
    62   65535.0    3929.5    3761.5    1085.7    1052.9    1026.6
    63   65535.0   65535.0   65535.0   65535.0   65535.0   65535.0

as loaded
               0         1         2        52        53        54
     0       nan       nan       nan       nan       nan       nan
     1       nan       nan       nan       nan       nan       nan
     2       nan       nan       nan       nan       nan       nan
    61       nan    3929.0    3761.0    1085.2    1052.4    1026.1
    62       nan    3929.5    3761.5    1085.7    1052.9    1026.6
    63       nan       nan       nan       nan       nan       nan


In this way, comparing the TRDR and the CDR permits to define the smile error and the exact wavelength of each band in a specific observation.

In [10]:
"""How far one band drifts in wavelength across the swath."""

for name, bands in (("l", 55), ("s", 19)):
    table = wavelengths.load(name, bands)
    print(f"{name}, {bands} bands")
    print(f"{'band':>6}{'min (nm)':>12}{'max (nm)':>12}{'spread':>10}")
    for band in edges(bands):
        column = table[:, band]
        if np.isnan(column).all():
            print(f"{band:>6}{'never calibrated':>34}")
            continue
        low, high = np.nanmin(column), np.nanmax(column)
        print(f"{band:>6}{low:>12.2f}{high:>12.2f}{high - low:>10.2f}")
    print()

l, 55 bands
  band    min (nm)    max (nm)    spread
     0                  never calibrated
     1     3923.10     3934.73     11.64
     2     3755.98     3769.45     13.47
    52     1079.66     1089.97     10.31
    53     1046.89     1057.23     10.34
    54     1020.68     1031.04     10.35

s, 19 bands
  band    min (nm)    max (nm)    spread
     0      372.60      377.70      5.10
     1      404.78      410.21      5.43
     2      436.98      442.72      5.74
    16      975.21      984.10      8.89
    17     1014.38     1023.37      8.99
    18     1046.96     1056.11      9.15



## Putting the two together

`bands_calibration.calibrate` does three things and nothing else. It reads the record for
this detector at this band count. It reverses the cube and the record together when the
record runs long wavelength first, so that every cube leaves in the same direction. And it
writes `NaN` over the columns and the bands the record never calibrated, so the flag never
survives as a number.

The shape does not change. A dead band stays a band, it just stops pretending to be data.

In [11]:
"""The same pixel before and after, and what each band is centred on."""

ordered, table = bands_calibration.calibrate(raw, "l")
centres = bands_calibration.centres(table)

print(f"{'band':>6}{'stored':>12}{'wavelength':>13}{'calibrated':>13}")
for band in edges(ordered.shape[2], 5):
    print(
        f"{band:>6}{raw[LINE, SAMPLE, band]:>12.3f}"
        f"{centres[band]:>13.1f}{ordered[LINE, SAMPLE, band]:>13.3f}"
    )

  band      stored   wavelength   calibrated
     0   65535.000       1023.6        0.250
     1       0.043       1049.8        0.236
     2       0.130       1082.6        0.245
     3       0.158       1154.7        0.243
     4       0.147       1213.7        0.230
    50       0.230       3506.4        0.147
    51       0.243       3639.6        0.158
    52       0.245       3759.5        0.130
    53       0.236       3926.3        0.043
    54       0.250          nan          nan


In [12]:
"""The cube after calibration, at the band the raw slice was printed at."""

print(f"cube {ordered.shape}, band 44, lines down, samples across")
block(ordered[:, :, 44], edges(ordered.shape[0]), edges(ordered.shape[1]), digits=3)

cube (2700, 64, 55), band 44, lines down, samples across
               0         1         2        61        62        63
     0       nan       nan       nan     0.173     0.145       nan
     1       nan       nan       nan     0.169     0.142       nan
     2       nan       nan       nan     0.167     0.143       nan
  2697       nan       nan       nan     0.183     0.154       nan
  2698       nan       nan       nan     0.186     0.157       nan
  2699       nan       nan       nan     0.185     0.156       nan


## What this leaves the rest of the pipeline

A calibrated cube is still lines by samples by bands, and a band is still an index. What
has changed is that the index now means the same thing in every observation of a given
configuration: the bands ascend in wavelength, and the wavelengths themselves are carried
beside the cube rather than assumed.

Two things are deliberately not done here and are worth stating.

**Nothing is resampled.** The wavelength of a band still depends on the sample it is read
at, because the smile is real and is up to 16 nm on the infrared detector. Comparing one
sample against another at a fixed band index compares slightly different colours. The
wavelengths are kept as a full table for exactly that reason, so any later step can see the
drift instead of averaging it away.

**Dead columns and bands are kept in place.** They are `NaN`, not removed. Dropping them
would change the shape of the cube per configuration and break the correspondence with the
geometry backplanes, which are on the same grid.

The remaining assumption is the record choice. `wavelengths.load` selects on detector and
band count, while the label names the record outright. That holds because survey mode has
only ever been seen naming the two configurations kept here, and it is checked above by
printing what each label names.